# Project: a quantum classifier versus a tiny network
**Quantum Machine Learning · Week 3 (Variational quantum classifiers) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/qml/w3d5.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

**Project: a quantum classifier versus a tiny network.** This week's best quantum classifier (three uploads, a
classical scale, cross-entropy, Adam) against a classical network with about as many weights. Same data, same
splits, same loss, one look at each test set, and an honest count of what each one costs.

In [ ]:
import time
import numpy as np
import torch
from torch import nn
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from sklearn.datasets import make_moons, make_circles
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

torch.set_num_threads(1)
estimator = AerEstimator(options={"backend_options": {"max_parallel_threads": 1}})   # exact ⟨Z⟩, faster than the reference one

def dataset(kind, seed):
    """120 points, 80 for training and 40 for the test, features scaled to [-π/2, π/2] using the training part only."""
    if kind == "moons":
        X, y = make_moons(120, noise=0.15, random_state=seed)
    elif kind == "circles":
        X, y = make_circles(120, noise=0.08, factor=0.5, random_state=seed)
    else:
        rng = np.random.default_rng(seed)
        corner = rng.choice([-1, 1], size=(120, 2))
        X, y = corner + 0.4 * rng.normal(size=(120, 2)), (corner[:, 0] * corner[:, 1] < 0).astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=40, random_state=seed, stratify=y)
    scaler = MinMaxScaler((-np.pi / 2, np.pi / 2)).fit(X_tr)
    return scaler.transform(X_tr), scaler.transform(X_te), y_tr, y_te

print({kind: [a.shape for a in dataset(kind, 0)[:2]] for kind in ("moons", "circles", "xor")})

## The quantum contestant
Thursday's re-uploading circuit (3 layers, 12 weights), Tuesday's classical scale $p = \sigma(-a\langle Z\rangle + b)$,
cross-entropy, Adam, and parameter-shift gradients: 1 + 2 × 12 = 25 circuit runs per point per epoch.

In [ ]:
x, w = ParameterVector("x", 2), ParameterVector("w", 12)
circuit = QuantumCircuit(2)
for l in range(3):
    circuit.ry(x[0], 0)
    circuit.ry(x[1], 1)
    circuit.ry(w[4 * l], 0)
    circuit.rz(w[4 * l + 1], 0)
    circuit.ry(w[4 * l + 2], 1)
    circuit.rz(w[4 * l + 3], 1)
    circuit.cx(0, 1)
Z1 = SparsePauliOp("ZI")

def scores(X, weight_sets):
    rows = np.array([np.concatenate([W, p]) for W in weight_sets for p in X])
    return estimator.run([(circuit, Z1, rows)]).result()[0].data.evs.reshape(len(weight_sets), len(X))

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def train_quantum(X, y, epochs=60, lr=0.1, seed=0):
    theta = np.r_[0.6 * np.random.default_rng(seed).normal(size=12), 2.0, 0.0]   # 12 circuit weights, then a and b
    m, v, runs = np.zeros(14), np.zeros(14), 0
    shifts = [s * np.eye(12)[j] for j in range(12) for s in (np.pi / 2, -np.pi / 2)]
    for k in range(1, epochs + 1):
        W, a, b = theta[:12], theta[12], theta[13]
        S = scores(X, [W] + [W + d for d in shifts])
        runs += S.size
        f, df = S[0], (S[1::2] - S[2::2]) / 2
        dz = (sigmoid(-a * f + b) - y) / len(y)                    # cross-entropy through the sigmoid
        g = np.r_[df @ (-a * dz), np.sum(-f * dz), np.sum(dz)]
        m, v = 0.9 * m + 0.1 * g, 0.999 * v + 0.001 * g ** 2
        theta = theta - lr * (m / (1 - 0.9 ** k)) / (np.sqrt(v / (1 - 0.999 ** k)) + 1e-8)
    predict = lambda X_: (sigmoid(-theta[12] * scores(X_, [theta[:12]])[0] + theta[13]) > 0.5).astype(int)
    return predict, runs

## The classical contestant
A network with 2 inputs, 3 tanh units and 1 output: 13 weights. Cross-entropy, Adam, 300 full-batch steps.

In [ ]:
def train_network(X, y, hidden=3, steps=300, lr=0.05, seed=0):
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(2, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)
    for step in range(steps):
        optimizer.zero_grad()
        nn.functional.binary_cross_entropy_with_logits(net(Xt).squeeze(1), yt).backward()
        optimizer.step()
    def predict(X_):
        with torch.no_grad():
            return (net(torch.tensor(X_, dtype=torch.float32)).squeeze(1) > 0).int().numpy()
    return predict, sum(p.numel() for p in net.parameters())

print("network weights:", train_network(*dataset("moons", 0)[::2], steps=1)[1])

## The duel: 3 datasets × 3 splits

In [ ]:
table = {}
for kind in ("moons", "circles", "xor"):
    q_acc, c_acc, q_time, c_time = [], [], 0.0, 0.0
    for seed in range(3):
        X_tr, X_te, y_tr, y_te = dataset(kind, seed)
        t0 = time.time()
        predict_q, runs = train_quantum(X_tr, y_tr, seed=seed)
        q_time += time.time() - t0
        t0 = time.time()
        predict_c, _ = train_network(X_tr, y_tr, seed=seed)
        c_time += time.time() - t0
        q_acc.append(np.mean(predict_q(X_te) == y_te))
        c_acc.append(np.mean(predict_c(X_te) == y_te))
    table[kind] = (q_acc, c_acc)
    print(f"{kind:8s} quantum {np.mean(q_acc):.3f} ± {np.std(q_acc):.3f} ({q_time / 3:4.1f} s per run)   "
          f"network {np.mean(c_acc):.3f} ± {np.std(c_acc):.3f} ({c_time / 3:.2f} s per run)")
print(f"circuit runs per quantum training: {runs:,}")

## Project
1. Read the table: where does each contestant win, and is any gap bigger than the spread?
2. Draw both decision boundaries for one split of the moons, side by side.
3. Count the bill: how many circuit runs did one quantum training take, and how many shots would that be on hardware
   with 1,000 shots per run? The network needed about 650,000 multiply-adds in all.
4. Write three honest sentences.

In [ ]:
import matplotlib.pyplot as plt

X_tr, X_te, y_tr, y_te = dataset("moons", 0)
# TODO: train both contestants on this split, draw their boundaries on a grid (two panels),
#       and print the quantum bill: circuit runs, and shots at 1,000 per run

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

X_tr, X_te, y_tr, y_te = dataset("moons", 0)
predict_q, runs = train_quantum(X_tr, y_tr)
predict_c, n_weights = train_network(X_tr, y_tr)
g = np.linspace(-2.0, 2.0, 61)
gx, gy = np.meshgrid(g, g)
grid = np.c_[gx.ravel(), gy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2), layout="constrained")
for ax, (name, predict) in zip(axes, [("quantum classifier: 14 weights", predict_q), (f"tiny network: {n_weights} weights", predict_c)]):
    ax.contourf(gx, gy, predict(grid).reshape(gx.shape), levels=[-0.5, 0.5, 1.5], cmap=ListedColormap(["#ffe1d2", "#d9e1ff"]))
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=np.where(y_tr == 0, "#ff8a50", "#5577f5"), s=18, edgecolors="white", linewidths=0.5)
    ax.scatter(X_te[:, 0], X_te[:, 1], facecolors="white", edgecolors=np.where(y_te == 0, "#ff8a50", "#5577f5"), s=26, linewidths=1.6)
    ax.set_title(f"{name}, test {np.mean(predict(X_te) == y_te):.3f}")
    ax.set_aspect("equal")
plt.show()
print(f"quantum bill: {runs:,} circuit runs = {runs * 1000:,} shots at 1,000 shots per run")
print(f"network bill: about {3 * 9 * 80 * 300:,} multiply-adds (forward and backward, 300 steps)")
print()
print("1. On these three small datasets the quantum classifier and a 13-weight network tie within the spread.")
print("2. The network trains in a fraction of a second on a laptop; the quantum model needs 120,000 circuit runs,")
print("   about 120 million shots on hardware, before counting noise.")
print("3. A tie on toy data says nothing about harder data either way; it does say that 'it works' is not evidence of an advantage.")